In [ ]:
# Import required libraries
import os
import time
import math
import scipy
import random
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Machine Learning
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import RobustScaler

# Deep Learning
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Hyperparameter Optimization
import optuna
from optuna.samplers import TPESampler

# Set random seed for reproducibility
RANDOM_SEED = 2025
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

warnings.filterwarnings("ignore")

# Configure logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)
logger = logging.getLogger("Record")

# Suppress Optuna logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Configuration
HISTORY_WINDOW = 12
PREDICTION_HORIZON = 6
NORMALIZE_PARA = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS_ML = 30
N_TRIALS_DL = 12
QUICK_EPOCHS = 20
FULL_EPOCHS = 80

logger.info(f"Using device: {DEVICE}")
logger.info(f"ML Models Trials: {N_TRIALS_ML}, DL Models Trials: {N_TRIALS_DL}")


def load_file(
    file_index: int = 1, normalize_para: int = 100, BW: int = 70, Ts: int = 5
):
    """
    Load and preprocess data file
    """

    data = scipy.io.loadmat(f"data/simulated/adult360_{file_index}.mat")
    data_history = data["history"]
    Glucose = data_history[0]["CGM"][0] / normalize_para
    Insulin = data_history[0]["u6"][0] / 6000 * BW
    Meal = data_history[0]["u1"][0] / 1000 * Ts
    num_days = len(Glucose) // 288
    day_index = np.array([x / 288 for x in range(288)] * num_days)
    day_index = day_index.reshape(-1, 1)
    data_full = np.concatenate((Glucose, Insulin, Meal, day_index), axis=1)
    df_full = pd.DataFrame(data=data_full, columns=["G", "I", "M", "T"])
    logger.info(f"Data adult360_{file_index}.mat loaded successfully.")
    logger.info(f"Data shape: {df_full.shape}")
    return df_full


class DataProcessor:
    """Class for processing diabetes time series data"""

    def __init__(self, df, history_window=12):
        self.df = df
        self.history_window = history_window
        self.data = df.values

    def create_supervised_dataset(self, prediction_horizon=6):
        """Convert time series data to supervised learning format"""
        X, y = [], []
        num_samples = len(self.data)

        for i in range(num_samples):
            end_ix = i + self.history_window
            out_ix = end_ix + prediction_horizon - 1

            if out_ix >= num_samples:
                break

            X.append(self.data[i:end_ix, :])
            y.append(self.data[out_ix, 0])

        return np.array(X), np.array(y)

    def split_data_chronological(self, X, y, train_ratio=0.6, val_ratio=0.2):
        """Split data chronologically into train, validation, and test sets"""
        n_samples = len(X)
        train_end = int(n_samples * train_ratio)
        val_end = int(n_samples * (train_ratio + val_ratio))

        X_train = X[:train_end]
        y_train = y[:train_end]
        X_val = X[train_end:val_end]
        y_val = y[train_end:val_end]
        X_test = X[val_end:]
        y_test = y[val_end:]

        return X_train, y_train, X_val, y_val, X_test, y_test


# Deep Learning Models
class LSTMModel(nn.Module):
    """
    LSTMModel
    """

    def __init__(
        self, input_size, hidden_size=64, num_layers=4, output_size=1, dropout=0.3
    ):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size),
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        output = self.fc(last_output)
        return output


class GRUModel(nn.Module):
    """
    GRUModel
    """

    def __init__(
        self, input_size, hidden_size=64, num_layers=4, output_size=1, dropout=0.3
    ):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size),
        )

    def forward(self, x):
        gru_out, _ = self.gru(x)
        last_output = gru_out[:, -1, :]
        output = self.fc(last_output)
        return output


class TransformerModel(nn.Module):
    """
    Transformer Model
    """

    def __init__(
        self, input_size, d_model, nhead, num_layers, output_size=1, dropout=0.3
    ):
        super(TransformerModel, self).__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers
        self.input_size = input_size

        # Input projection to d_model dimension
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, d_model),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )

        # Positional encoding
        self.register_buffer(
            "positional_encoding",
            self._create_sinusoidal_embeddings(100, d_model),
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 3,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )

        self.use_avg_pool = True

        # Output layers
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.LayerNorm(d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, output_size),
        )

        self.dropout = nn.Dropout(dropout)

    def _create_sinusoidal_embeddings(self, max_len, d_model):
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-np.log(10000.0) / d_model)
        )
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, x):
        _, seq_len, _ = x.shape

        # Project input to d_model dimension
        x = self.input_projection(x)

        # Add positional encoding
        x = x + self.positional_encoding[:, :seq_len, :]
        x = self.dropout(x)

        # Transformer encoding
        transformer_out = self.transformer_encoder(x)

        # Pooling
        if self.use_avg_pool:
            avg_pool = torch.mean(transformer_out, dim=1)
            last_step = transformer_out[:, -1, :]
            combined = (avg_pool + last_step) / 2
        else:
            combined = transformer_out[:, -1, :]

        # Output
        output = self.fc(combined)
        return output


def worker_init_fn(worker_id):
    """Initialize worker with different random seed for reproducibility"""
    worker_seed = RANDOM_SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def train_deep_model_dynamic(
    model,
    train_loader,
    val_loader,
    epochs=100,
    lr=5e-4,
    patience=15,
):
    """
    Train deep learning model with dynamic GPU loading and improved stability

    Args:
        model: PyTorch model
        train_loader: Training data loader
        val_loader: Validation data loader
        epochs: Maximum number of epochs
        lr: Learning rate
        patience: Early stopping patience
    """

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-8
    )

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for inputs, labels in train_loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

            del inputs, labels, outputs, loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)

                del inputs, labels, outputs, loss

        train_loss /= len(train_loader.dataset)
        val_loss /= len(val_loader.dataset)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_model_state)
    return model, np.sqrt(best_val_loss)


def evaluate_model(model, X_test, y_test, is_ml=False, normalize_para=100):
    """Evaluate model and return metrics"""
    if is_ml:
        y_pred = model.predict(X_test)
    else:
        model.eval()
        device = next(model.parameters()).device
        X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
        with torch.no_grad():
            y_pred = model(X_test_t).cpu().numpy().flatten()

    y_pred_real = y_pred * normalize_para
    y_test_real = y_test * normalize_para

    rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
    mae = mean_absolute_error(y_test_real, y_pred_real)
    r2 = r2_score(y_test_real, y_pred_real)

    return {"RMSE": rmse, "MAE": mae, "R2": r2}, y_pred


def optimize_ridge(X_train, y_train, X_val, y_val):
    """Optimize Ridge hyperparameters"""

    def objective(trial):
        alpha = trial.suggest_float("alpha", 0.01, 100.0, log=True)
        model = Ridge(alpha=alpha, random_state=RANDOM_SEED)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        return rmse * NORMALIZE_PARA

    study = optuna.create_study(
        direction="minimize", sampler=TPESampler(seed=RANDOM_SEED)
    )
    study.optimize(objective, n_trials=N_TRIALS_ML, show_progress_bar=True)

    best_model = Ridge(alpha=study.best_params["alpha"], random_state=RANDOM_SEED)
    best_model.fit(X_train, y_train)
    return best_model, study.best_value


def optimize_random_forest(X_train, y_train, X_val, y_val):
    """Optimize Random Forest hyperparameters"""

    def objective(trial):
        n_estimators = trial.suggest_int("n_estimators", 50, 300)
        max_depth = trial.suggest_int("max_depth", 5, 30)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)

        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        return rmse * NORMALIZE_PARA

    study = optuna.create_study(
        direction="minimize", sampler=TPESampler(seed=RANDOM_SEED)
    )
    study.optimize(objective, n_trials=N_TRIALS_ML, show_progress_bar=True)

    best_model = RandomForestRegressor(
        n_estimators=study.best_params["n_estimators"],
        max_depth=study.best_params["max_depth"],
        min_samples_split=study.best_params["min_samples_split"],
        min_samples_leaf=study.best_params["min_samples_leaf"],
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    best_model.fit(X_train, y_train)
    return best_model, study.best_value


def optimize_lightgbm(X_train, y_train, X_val, y_val):
    """Optimize LightGBM hyperparameters"""

    def objective(trial):
        params = {
            "objective": "regression",
            "metric": "rmse",
            "verbosity": -1,
            "boosting_type": "gbdt",
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 20, 150),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "random_state": RANDOM_SEED,
        }

        model = lgb.LGBMRegressor(**params, n_estimators=500)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        return rmse * NORMALIZE_PARA

    study = optuna.create_study(
        direction="minimize", sampler=TPESampler(seed=RANDOM_SEED)
    )
    study.optimize(objective, n_trials=N_TRIALS_ML, show_progress_bar=True)

    best_params = study.best_params
    best_params.update(
        {
            "objective": "regression",
            "metric": "rmse",
            "verbosity": -1,
            "boosting_type": "gbdt",
            "random_state": RANDOM_SEED,
        }
    )

    best_model = lgb.LGBMRegressor(**best_params, n_estimators=500)
    best_model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )
    return best_model, study.best_value


def optimize_deep_model(
    model_class,
    model_name,
    X_train,
    y_train,
    X_val,
    y_val,
    input_size,
    n_trials=10,
    quick_epochs=20,
    full_epochs=80,
):
    """
    Universal deep learning model optimization function

    Args:
        model_class: Model class
        model_name: Model name
        X_train, y_train, X_val, y_val: Training and validation data
        input_size: Input feature dimension
        n_trials: Number of Optuna optimization trials
        quick_epochs: Number of epochs for quick validation
        full_epochs: Number of epochs for full training
    """

    def objective(trial):
        if model_name == "Transformer":
            valid_configs = [(32, 4), (64, 4), (64, 8), (128, 4), (128, 8)]
            config_idx = trial.suggest_categorical(
                "config_idx", list(range(len(valid_configs)))
            )
            d_model, nhead = valid_configs[config_idx]
            num_layers = trial.suggest_int("num_layers", 2, 4)
            dropout = trial.suggest_float("dropout", 0.1, 0.5)

            model = model_class(
                input_size=input_size,
                d_model=d_model,
                nhead=nhead,
                num_layers=num_layers,
                dropout=dropout,
            ).to(DEVICE)
        else:
            hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
            num_layers = trial.suggest_int("num_layers", 2, 4)
            dropout = trial.suggest_float("dropout", 0.1, 0.5)

            model = model_class(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                dropout=dropout,
            ).to(DEVICE)

        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
        X_val_t = torch.tensor(X_val, dtype=torch.float32)
        y_val_t = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)

        train_dataset = TensorDataset(X_train_t, y_train_t)
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            worker_init_fn=worker_init_fn,
            pin_memory=True,
            num_workers=0,
        )
        val_dataset = TensorDataset(X_val_t, y_val_t)
        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            pin_memory=True,
            num_workers=0,
        )

        _, val_rmse = train_deep_model_dynamic(
            model, train_loader, val_loader, epochs=quick_epochs, lr=lr, patience=8
        )

        return val_rmse * NORMALIZE_PARA

    study = optuna.create_study(
        direction="minimize", sampler=TPESampler(seed=RANDOM_SEED)
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    # Full training with best parameters
    best_params = study.best_params

    if model_name == "Transformer":
        valid_configs = [(32, 4), (64, 4), (64, 8), (128, 4), (128, 8)]
        d_model, nhead = valid_configs[best_params["config_idx"]]

        best_model = model_class(
            input_size=input_size,
            d_model=d_model,
            nhead=nhead,
            num_layers=best_params["num_layers"],
            dropout=best_params["dropout"],
        ).to(DEVICE)
    else:
        best_model = model_class(
            input_size=input_size,
            hidden_size=best_params["hidden_size"],
            num_layers=best_params["num_layers"],
            dropout=best_params["dropout"],
        ).to(DEVICE)

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)

    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(
        train_dataset,
        batch_size=best_params["batch_size"],
        shuffle=True,
        worker_init_fn=worker_init_fn,
        pin_memory=True,
        num_workers=0,
    )
    val_dataset = TensorDataset(X_val_t, y_val_t)
    val_loader = DataLoader(
        val_dataset,
        batch_size=best_params["batch_size"],
        shuffle=False,
        pin_memory=True,
        num_workers=0,
    )

    best_model, final_val_rmse = train_deep_model_dynamic(
        best_model,
        train_loader,
        val_loader,
        epochs=full_epochs,
        lr=best_params["lr"],
        patience=15,
    )

    return best_model, final_val_rmse * NORMALIZE_PARA


def optimize_lstm(X_train, y_train, X_val, y_val, input_size):
    return optimize_deep_model(
        LSTMModel,
        "LSTM",
        X_train,
        y_train,
        X_val,
        y_val,
        input_size,
        n_trials=N_TRIALS_DL,
        quick_epochs=QUICK_EPOCHS,
        full_epochs=FULL_EPOCHS,
    )


def optimize_gru(X_train, y_train, X_val, y_val, input_size):
    return optimize_deep_model(
        GRUModel,
        "GRU",
        X_train,
        y_train,
        X_val,
        y_val,
        input_size,
        n_trials=N_TRIALS_DL,
        quick_epochs=QUICK_EPOCHS,
        full_epochs=FULL_EPOCHS,
    )


def optimize_transformer(X_train, y_train, X_val, y_val, input_size):
    return optimize_deep_model(
        TransformerModel,
        "Transformer",
        X_train,
        y_train,
        X_val,
        y_val,
        input_size,
        n_trials=N_TRIALS_DL,
        quick_epochs=QUICK_EPOCHS,
        full_epochs=FULL_EPOCHS,
    )


def scale_3d_data(X_data, scaler):
    """
    对3D数据进行特征级标准化

    Args:
        X_data: (n_samples, n_steps, n_features)
        scaler: 已经fit的RobustScaler

    Returns:
        标准化后的3D数组 (n_samples, n_steps, n_features)
    """

    n_samples, n_steps, n_features = X_data.shape
    X_reshaped = X_data.reshape(-1, n_features)
    X_scaled = scaler.transform(X_reshaped)

    return X_scaled.reshape(n_samples, n_steps, n_features)


logger.info("=" * 80)
logger.info("Starting Data Collection for Global Normalization")
logger.info("=" * 80)

# Step 1: Collect all training data from all files
all_data_collection = {}

for file_idx in [1, 2, 3]:
    logger.info(f"\n--- Collecting data from File {file_idx} ---")
    df = load_file(file_idx)
    processor = DataProcessor(df, history_window=HISTORY_WINDOW)
    X, y = processor.create_supervised_dataset(prediction_horizon=PREDICTION_HORIZON)
    X_train, y_train, X_val, y_val, X_test, y_test = processor.split_data_chronological(
        X, y, train_ratio=0.6, val_ratio=0.2
    )

    all_data_collection[file_idx] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
    }

# Step 2: Fit unified global scaler
logger.info("\n" + "=" * 80)
logger.info("Fitting Unified Global Scaler")
logger.info("=" * 80)

all_X_train_concat = np.concatenate(
    [all_data_collection[i]["X_train"] for i in [1, 2, 3]], axis=0
)

nsamples, nsteps, nfeatures = all_X_train_concat.shape
X_train_reshaped = all_X_train_concat.reshape(-1, nfeatures)

global_scaler = RobustScaler()
global_scaler.fit(X_train_reshaped)

logger.info("Unified global scaler fitted successfully")
logger.info(
    f"Scaler statistics: center={global_scaler.center_}, scale={global_scaler.scale_}"
)

# ============================================================================
# Main Training Loop
# ============================================================================

logger.info("\n" + "=" * 80)
logger.info("Starting Model Training and Evaluation")
logger.info("=" * 80)

all_results = {
    "Ridge": {"RMSE": [], "MAE": [], "R2": [], "predictions": []},
    "Random Forest": {"RMSE": [], "MAE": [], "R2": [], "predictions": []},
    "LightGBM": {"RMSE": [], "MAE": [], "R2": [], "predictions": []},
    "LSTM": {"RMSE": [], "MAE": [], "R2": [], "predictions": []},
    "GRU": {"RMSE": [], "MAE": [], "R2": [], "predictions": []},
    "Transformer": {"RMSE": [], "MAE": [], "R2": [], "predictions": []},
}

file_test_data = {}

for file_idx in [1, 2, 3]:
    logger.info("\n" + "=" * 80)
    logger.info(f"Processing File {file_idx}")
    logger.info("=" * 80)

    # Use previously collected data
    X_train = all_data_collection[file_idx]["X_train"]
    y_train = all_data_collection[file_idx]["y_train"]
    X_val = all_data_collection[file_idx]["X_val"]
    y_val = all_data_collection[file_idx]["y_val"]
    X_test = all_data_collection[file_idx]["X_test"]
    y_test = all_data_collection[file_idx]["y_test"]

    X_train_scaled_dl = scale_3d_data(X_train, global_scaler)
    X_val_scaled_dl = scale_3d_data(X_val, global_scaler)
    X_test_scaled_dl = scale_3d_data(X_test, global_scaler)

    X_train_scaled_3d = scale_3d_data(X_train, global_scaler)
    X_train_ml_scaled = X_train_scaled_3d.reshape(len(X_train), -1)

    X_val_scaled_3d = scale_3d_data(X_val, global_scaler)
    X_val_ml_scaled = X_val_scaled_3d.reshape(len(X_val), -1)

    X_test_scaled_3d = scale_3d_data(X_test, global_scaler)
    X_test_ml_scaled = X_test_scaled_3d.reshape(len(X_test), -1)

    input_size = X_train.shape[2]

    logger.info(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    logger.info(f"Input size: {input_size}")

    # 🔧 Store test data with consistent variable names
    file_test_data[file_idx] = {
        "y_test": y_test,
        "X_test": X_test,
        "X_test_scaled_dl": X_test_scaled_dl,
        "X_test_ml_scaled": X_test_ml_scaled,
    }

    # ========================================================================
    # Ridge Regression
    # ========================================================================
    logger.info("\n--- Training Ridge Regression ---")
    ridge_model, ridge_val_loss = optimize_ridge(
        X_train_ml_scaled, y_train, X_val_ml_scaled, y_val
    )
    logger.info(f"Ridge - Best Validation RMSE: {ridge_val_loss:.6f} mg/dL")
    ridge_metrics, ridge_pred = evaluate_model(
        ridge_model, X_test_ml_scaled, y_test, is_ml=True
    )
    logger.info(
        f"Ridge Test - RMSE: {ridge_metrics['RMSE']:.2f}, MAE: {ridge_metrics['MAE']:.2f}, R2: {ridge_metrics['R2']:.4f}"
    )
    all_results["Ridge"]["RMSE"].append(ridge_metrics["RMSE"])
    all_results["Ridge"]["MAE"].append(ridge_metrics["MAE"])
    all_results["Ridge"]["R2"].append(ridge_metrics["R2"])
    all_results["Ridge"]["predictions"].append(ridge_pred)

    # ========================================================================
    # Random Forest
    # ========================================================================
    logger.info("\n--- Training Random Forest ---")
    rf_model, rf_val_loss = optimize_random_forest(
        X_train_ml_scaled, y_train, X_val_ml_scaled, y_val
    )
    logger.info(f"Random Forest - Best Validation RMSE: {rf_val_loss:.6f} mg/dL")
    rf_metrics, rf_pred = evaluate_model(rf_model, X_test_ml_scaled, y_test, is_ml=True)
    logger.info(
        f"Random Forest Test - RMSE: {rf_metrics['RMSE']:.2f}, MAE: {rf_metrics['MAE']:.2f}, R2: {rf_metrics['R2']:.4f}"
    )
    all_results["Random Forest"]["RMSE"].append(rf_metrics["RMSE"])
    all_results["Random Forest"]["MAE"].append(rf_metrics["MAE"])
    all_results["Random Forest"]["R2"].append(rf_metrics["R2"])
    all_results["Random Forest"]["predictions"].append(rf_pred)

    # ========================================================================
    # LightGBM
    # ========================================================================
    logger.info("\n--- Training LightGBM ---")
    lgb_model, lgb_val_loss = optimize_lightgbm(
        X_train_ml_scaled, y_train, X_val_ml_scaled, y_val
    )
    logger.info(f"LightGBM - Best Validation RMSE: {lgb_val_loss:.6f} mg/dL")
    lgb_metrics, lgb_pred = evaluate_model(
        lgb_model, X_test_ml_scaled, y_test, is_ml=True
    )
    logger.info(
        f"LightGBM Test - RMSE: {lgb_metrics['RMSE']:.2f}, MAE: {lgb_metrics['MAE']:.2f}, R2: {lgb_metrics['R2']:.4f}"
    )
    all_results["LightGBM"]["RMSE"].append(lgb_metrics["RMSE"])
    all_results["LightGBM"]["MAE"].append(lgb_metrics["MAE"])
    all_results["LightGBM"]["R2"].append(lgb_metrics["R2"])
    all_results["LightGBM"]["predictions"].append(lgb_pred)

    # ========================================================================
    # LSTM
    # ========================================================================
    logger.info("\n--- Training LSTM ---")
    lstm_model, lstm_val_loss = optimize_lstm(
        X_train_scaled_dl, y_train, X_val_scaled_dl, y_val, input_size
    )
    logger.info(f"LSTM - Best Validation RMSE: {lstm_val_loss:.6f} mg/dL")
    lstm_metrics, lstm_pred = evaluate_model(
        lstm_model, X_test_scaled_dl, y_test, is_ml=False
    )
    logger.info(
        f"LSTM Test - RMSE: {lstm_metrics['RMSE']:.2f}, MAE: {lstm_metrics['MAE']:.2f}, R2: {lstm_metrics['R2']:.4f}"
    )
    all_results["LSTM"]["RMSE"].append(lstm_metrics["RMSE"])
    all_results["LSTM"]["MAE"].append(lstm_metrics["MAE"])
    all_results["LSTM"]["R2"].append(lstm_metrics["R2"])
    all_results["LSTM"]["predictions"].append(lstm_pred)

    # ========================================================================
    # GRU
    # ========================================================================
    logger.info("\n--- Training GRU ---")
    gru_model, gru_val_loss = optimize_gru(
        X_train_scaled_dl, y_train, X_val_scaled_dl, y_val, input_size
    )
    logger.info(f"GRU - Best Validation RMSE: {gru_val_loss:.6f} mg/dL")
    gru_metrics, gru_pred = evaluate_model(
        gru_model, X_test_scaled_dl, y_test, is_ml=False
    )
    logger.info(
        f"GRU Test - RMSE: {gru_metrics['RMSE']:.2f}, MAE: {gru_metrics['MAE']:.2f}, R2: {gru_metrics['R2']:.4f}"
    )
    all_results["GRU"]["RMSE"].append(gru_metrics["RMSE"])
    all_results["GRU"]["MAE"].append(gru_metrics["MAE"])
    all_results["GRU"]["R2"].append(gru_metrics["R2"])
    all_results["GRU"]["predictions"].append(gru_pred)

    # ========================================================================
    # Transformer
    # ========================================================================
    logger.info("\n--- Training Transformer ---")
    transformer_model, transformer_val_loss = optimize_transformer(
        X_train_scaled_dl, y_train, X_val_scaled_dl, y_val, input_size
    )
    logger.info(f"Transformer - Best Validation RMSE: {transformer_val_loss:.6f} mg/dL")
    transformer_metrics, transformer_pred = evaluate_model(
        transformer_model, X_test_scaled_dl, y_test, is_ml=False
    )
    logger.info(
        f"Transformer Test - RMSE: {transformer_metrics['RMSE']:.2f}, MAE: {transformer_metrics['MAE']:.2f}, R2: {transformer_metrics['R2']:.4f}"
    )
    all_results["Transformer"]["RMSE"].append(transformer_metrics["RMSE"])
    all_results["Transformer"]["MAE"].append(transformer_metrics["MAE"])
    all_results["Transformer"]["R2"].append(transformer_metrics["R2"])
    all_results["Transformer"]["predictions"].append(transformer_pred)

In [ ]:
# ============================================================================
# Calculate Statistics and Print Final Results
# ============================================================================

logger.info("\n" + "=" * 80)
logger.info("FINAL MODEL PERFORMANCE SUMMARY (Mean ± Std)")
logger.info("=" * 80)

summary_data = []
for model_name, results in all_results.items():
    rmse_mean = np.mean(results["RMSE"])
    rmse_std = np.std(results["RMSE"])
    mae_mean = np.mean(results["MAE"])
    mae_std = np.std(results["MAE"])
    r2_mean = np.mean(results["R2"])
    r2_std = np.std(results["R2"])

    summary_data.append(
        {
            "Model": model_name,
            "RMSE": f"{rmse_mean:.2f} ± {rmse_std:.2f}",
            "MAE": f"{mae_mean:.2f} ± {mae_std:.2f}",
            "R2": f"{r2_mean:.4f} ± {r2_std:.4f}",
        }
    )

    logger.info(f"\n{model_name}:")
    logger.info(f"  RMSE: {rmse_mean:.2f} ± {rmse_std:.2f} mg/dL")
    logger.info(f"  MAE:  {mae_mean:.2f} ± {mae_std:.2f} mg/dL")
    logger.info(f"  R2:   {r2_mean:.4f} ± {r2_std:.4f}")

# Create summary DataFrame
summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 80)
print("MODEL PERFORMANCE TABLE")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

In [ ]:
# ============================================================================
# Plot Predictions for Patient 1
# ============================================================================


def plot_all_predictions(all_results, file_test_data, num_points=600):
    """Plot true glucose and predictions from all models for Patient 1"""

    y_true = file_test_data[1]["y_test"][:num_points]

    plt.figure(figsize=(20, 10), dpi=300)

    # Plot true glucose values
    plt.plot(
        y_true * NORMALIZE_PARA,
        label="True Glucose",
        color="black",
        linewidth=2.5,
        alpha=0.8,
    )

    # Define colors and line styles
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
    linestyles = ["-", "--", "-.", ":", "-", "--"]

    # Plot predictions from each model
    for idx, (model_name, results) in enumerate(all_results.items()):
        y_pred = results["predictions"][0][:num_points]
        plt.plot(
            y_pred * NORMALIZE_PARA,
            label=f"{model_name}",
            color=colors[idx],
            linestyle=linestyles[idx],
            linewidth=1.8,
            alpha=0.7,
        )

    plt.title(
        f"Glucose Prediction Comparison - Patient 1 Test Set (First {num_points} time points)",
        fontsize=48,
        fontweight="bold",
    )
    plt.xlabel("Time (5-minute intervals)", fontsize=42)
    plt.ylabel("Glucose (mg/dL)", fontsize=42)
    plt.legend(loc="upper right", fontsize=33, framealpha=0.9)
    plt.grid(True, alpha=0.3, linestyle="--")
    plt.margins(x=0.01, y=0.05)
    plt.tight_layout()

    plt.savefig("glucose_prediction_patient_1.png", dpi=300, bbox_inches="tight")
    plt.show()

    logger.info("\nPrediction plot for Patient 1 saved successfully.")


# Generate the prediction plot
plot_all_predictions(all_results, file_test_data, num_points=600)

logger.info("\n" + "=" * 80)
logger.info("ALL EXPERIMENTS COMPLETED SUCCESSFULLY")
logger.info("=" * 80)

In [ ]:
import pickle
import json

# ============================================================================
# Save Results to Disk
# ============================================================================

logger.info("\n" + "=" * 80)
logger.info("Saving Results to Disk")
logger.info("=" * 80)

# Create results directory if it doesn't exist
results_dir = "./results"
os.makedirs(results_dir, exist_ok=True)

# ============================================================================
# 1. Save using Pickle (Preserves all Python objects)
# ============================================================================

pickle_file = os.path.join(results_dir, "all_results.pkl")
with open(pickle_file, "wb") as f:
    pickle.dump(all_results, f)
logger.info(f"all_results saved to: {pickle_file}")

pickle_test_file = os.path.join(results_dir, "file_test_data.pkl")
with open(pickle_test_file, "wb") as f:
    pickle.dump(file_test_data, f)
logger.info(f"file_test_data saved to: {pickle_test_file}")

# ============================================================================
# 2. Save Metrics Summary as CSV
# ============================================================================

metrics_summary = []
for file_idx in [1, 2, 3]:
    for model_name, results in all_results.items():
        metrics_summary.append(
            {
                "File": file_idx,
                "Model": model_name,
                "RMSE": results["RMSE"][file_idx - 1],
                "MAE": results["MAE"][file_idx - 1],
                "R2": results["R2"][file_idx - 1],
            }
        )

metrics_df = pd.DataFrame(metrics_summary)
csv_file = os.path.join(results_dir, "metrics_summary.csv")
metrics_df.to_csv(csv_file, index=False)
logger.info(f"Metrics summary saved to: {csv_file}")

# ============================================================================
# 3. Save Predictions as CSV (for each patient)
# ============================================================================

for file_idx in [1, 2, 3]:
    predictions_data = {"y_true": file_test_data[file_idx]["y_test"] * NORMALIZE_PARA}

    for model_name, results in all_results.items():
        predictions_data[model_name] = (
            results["predictions"][file_idx - 1] * NORMALIZE_PARA
        )

    pred_df = pd.DataFrame(predictions_data)
    pred_csv_file = os.path.join(results_dir, f"predictions_patient_{file_idx}.csv")
    pred_df.to_csv(pred_csv_file, index=False)
    logger.info(f"Patient {file_idx} predictions saved to: {pred_csv_file}")

# ============================================================================
# 4. Save Test Data Numpy Arrays
# ============================================================================

for file_idx in [1, 2, 3]:
    test_data_dir = os.path.join(results_dir, f"patient_{file_idx}_test_data")
    os.makedirs(test_data_dir, exist_ok=True)

    np.save(
        os.path.join(test_data_dir, "y_test.npy"), file_test_data[file_idx]["y_test"]
    )
    np.save(
        os.path.join(test_data_dir, "X_test.npy"), file_test_data[file_idx]["X_test"]
    )
    np.save(
        os.path.join(test_data_dir, "X_test_scaled_dl.npy"),
        file_test_data[file_idx]["X_test_scaled_dl"],
    )
    np.save(
        os.path.join(test_data_dir, "X_test_ml_scaled.npy"),
        file_test_data[file_idx]["X_test_ml_scaled"],
    )

    logger.info(f"Patient {file_idx} test data arrays saved to: {test_data_dir}")

# ============================================================================
# 5. Save Detailed Summary Report (JSON)
# ============================================================================

summary_report = {
    "experiment_info": {
        "history_window": HISTORY_WINDOW,
        "prediction_horizon": PREDICTION_HORIZON,
        "normalize_para": NORMALIZE_PARA,
        "random_seed": RANDOM_SEED,
        "ml_trials": N_TRIALS_ML,
        "dl_trials": N_TRIALS_DL,
        "quick_epochs": QUICK_EPOCHS,
        "full_epochs": FULL_EPOCHS,
        "device": str(DEVICE),
    },
    "model_statistics": {},
}

for model_name, results in all_results.items():
    summary_report["model_statistics"][model_name] = {
        "rmse": {
            "mean": float(np.mean(results["RMSE"])),
            "std": float(np.std(results["RMSE"])),
            "values": [float(x) for x in results["RMSE"]],
        },
        "mae": {
            "mean": float(np.mean(results["MAE"])),
            "std": float(np.std(results["MAE"])),
            "values": [float(x) for x in results["MAE"]],
        },
        "r2": {
            "mean": float(np.mean(results["R2"])),
            "std": float(np.std(results["R2"])),
            "values": [float(x) for x in results["R2"]],
        },
    }

json_file = os.path.join(results_dir, "experiment_summary.json")
with open(json_file, "w") as f:
    json.dump(summary_report, f, indent=4)
logger.info(f"Experiment summary report saved to: {json_file}")

# ============================================================================
# 6. Load Results Back (Verification) - 修复日志格式错误
# ============================================================================

logger.info("\n" + "=" * 80)
logger.info("Verification: Loading Results from Disk")
logger.info("=" * 80)

# Load from pickle
with open(pickle_file, "rb") as f:
    loaded_all_results = pickle.load(f)

with open(pickle_test_file, "rb") as f:
    loaded_file_test_data = pickle.load(f)

# Verify structure (修复: 使用正确的格式化方式)
logger.info(f"Models in loaded results: {list(loaded_all_results.keys())}")
logger.info(f"Patient indices in test data: {list(loaded_file_test_data.keys())}")

# Verify data integrity
for model_name in loaded_all_results:
    rmse_values = loaded_all_results[model_name]["RMSE"]
    logger.info(
        f"{model_name} RMSE values (3 patients): {[f'{x:.2f}' for x in rmse_values]}"
    )

logger.info("\n" + "=" * 80)
logger.info("All Results Saved and Verified Successfully!")
logger.info(f"Results directory: {os.path.abspath(results_dir)}")
logger.info("=" * 80)

# ============================================================================
# 7. 额外功能: 保存模型汇总统计表
# ============================================================================

# 创建更详细的汇总表
detailed_summary = []
for model_name, results in all_results.items():
    detailed_summary.append(
        {
            "Model": model_name,
            "RMSE_Mean": f"{np.mean(results['RMSE']):.2f}",
            "RMSE_Std": f"{np.std(results['RMSE']):.2f}",
            "RMSE_Patient1": f"{results['RMSE'][0]:.2f}",
            "RMSE_Patient2": f"{results['RMSE'][1]:.2f}",
            "RMSE_Patient3": f"{results['RMSE'][2]:.2f}",
            "MAE_Mean": f"{np.mean(results['MAE']):.2f}",
            "R2_Mean": f"{np.mean(results['R2']):.4f}",
        }
    )

detailed_summary_df = pd.DataFrame(detailed_summary)
detailed_csv = os.path.join(results_dir, "detailed_model_summary.csv")
detailed_summary_df.to_csv(detailed_csv, index=False)
logger.info(f"Detailed model summary saved to: {detailed_csv}")


In [ ]:
# 方法1: 加载完整对象
import pickle

with open("results/all_results.pkl", "rb") as f:
    all_results = pickle.load(f)

# 方法2: 加载CSV表格
import pandas as pd

metrics_df = pd.read_csv("results/metrics_summary.csv")
predictions_df = pd.read_csv("results/predictions_patient_1.csv")

# 方法3: 加载NumPy数组
import numpy as np

y_test = np.load("results/patient_1_test_data/y_test.npy")

In [ ]:
# ============================================================================
# Plot Predictions for Patient 1 (Improved Version)
# ============================================================================


def plot_all_predictions(all_results, file_test_data, num_points=600):
    """
    Plot true glucose and predictions from all models for Patient 1
    with enlarged axis labels and legend positioned outside the plot area
    """

    y_true = file_test_data[1]["y_test"][:num_points]

    # Create figure with extra space for external legend
    fig, ax = plt.subplots(figsize=(24, 12), dpi=300)

    # Plot true glucose values
    ax.plot(
        y_true * NORMALIZE_PARA,
        label="True Glucose",
        color="black",
        linewidth=3.0,
        alpha=0.9,
        zorder=5,
    )

    # Define colors and line styles
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
    linestyles = ["-", "--", "-.", ":", "-", "--"]

    # Plot predictions from each model
    for idx, (model_name, results) in enumerate(all_results.items()):
        y_pred = results["predictions"][0][:num_points]
        ax.plot(
            y_pred * NORMALIZE_PARA,
            label=f"{model_name}",
            color=colors[idx],
            linestyle=linestyles[idx],
            linewidth=2.2,
            alpha=0.75,
            zorder=4 - idx * 0.1,
        )

    # 🔧 设置标题和轴标签 - 大尺寸字体
    ax.set_title(
        f"Glucose Prediction Comparison - Patient 1 Test Set (First {num_points} time points)",
        fontsize=30,
        fontweight="bold",
        pad=30,
    )

    ax.set_xlabel(
        "Time (5-minute intervals)", fontsize=25, fontweight="bold", labelpad=20
    )
    ax.set_ylabel("Glucose (mg/dL)", fontsize=25, fontweight="bold", labelpad=20)

    # 🔧 放大刻度标签 (tick labels)
    ax.tick_params(axis="x", labelsize=40, length=12, width=2.5)
    ax.tick_params(axis="y", labelsize=40, length=12, width=2.5)

    # 🔧 加粗坐标轴线
    for spine in ax.spines.values():
        spine.set_linewidth(3)

    # 🔧 网格设置
    ax.grid(True, alpha=0.3, linestyle="--", linewidth=1.5)

    # 🔧 设置图例到右上外侧 - 避免遮挡
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),  # 图例放在右侧外面
        fontsize=38,
        framealpha=0.95,
        fancybox=True,
        shadow=True,
        frameon=True,
        borderpad=1.5,
    )

    # 设置边距
    plt.margins(x=0.01, y=0.05)

    # 紧凑布局 - 考虑外侧图例
    plt.tight_layout()

    # 保存图片
    plt.savefig(
        "glucose_prediction_patient_1.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none",
    )
    plt.show()

    logger.info("\nPrediction plot for Patient 1 saved successfully.")


# Generate the prediction plot
plot_all_predictions(all_results, file_test_data, num_points=600)

In [ ]:
# ============================================================================
# Plot Predictions for Patient 1 (Improved Version)
# ============================================================================


def plot_all_predictions(all_results, file_test_data, num_points=600):
    """
    Plot true glucose and predictions from all models for Patient 1
    with enlarged axis labels and legend positioned outside the plot area
    """

    y_true = file_test_data[1]["y_test"][:num_points]

    # Create figure with extra space for external legend
    fig, ax = plt.subplots(figsize=(24, 12), dpi=300)

    # Plot true glucose values
    ax.plot(
        y_true * NORMALIZE_PARA,
        label="True Glucose",
        color="black",
        linewidth=3.0,
        alpha=0.9,
        zorder=5,
    )

    # Define colors and line styles
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
    linestyles = ["-", "--", "-.", ":", "-", "--"]

    # Plot predictions from each model
    for idx, (model_name, results) in enumerate(all_results.items()):
        y_pred = results["predictions"][0][:num_points]
        ax.plot(
            y_pred * NORMALIZE_PARA,
            label=f"{model_name}",
            color=colors[idx],
            linestyle=linestyles[idx],
            linewidth=2.2,
            alpha=0.75,
            zorder=4 - idx * 0.1,
        )

    # 🔧 设置标题和轴标签 - 大尺寸字体
    ax.set_title(
        f"Glucose Prediction Comparison - Patient 1 Test Set (First {num_points} time points)",
        fontsize=30,
        fontweight="bold",
        pad=30,
    )

    ax.set_xlabel(
        "Time (5-minute intervals)", fontsize=25, fontweight="bold", labelpad=20
    )
    ax.set_ylabel("Glucose (mg/dL)", fontsize=25, fontweight="bold", labelpad=20)

    # 🔧 放大刻度标签 (tick labels)
    ax.tick_params(axis="x", labelsize=40, length=12, width=2.5)
    ax.tick_params(axis="y", labelsize=40, length=12, width=2.5)

    # 🔧 加粗坐标轴线
    for spine in ax.spines.values():
        spine.set_linewidth(3)

    # 🔧 网格设置
    ax.grid(True, alpha=0.3, linestyle="--", linewidth=1.5)

    # 🔧 设置图例到右上外侧 - 避免遮挡
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),  # 图例放在右侧外面
        fontsize=38,
        framealpha=0.95,
        fancybox=True,
        shadow=True,
        frameon=True,
        borderpad=1.5,
    )

    # 设置边距
    plt.margins(x=0.01, y=0.05)

    # 紧凑布局 - 考虑外侧图例
    plt.tight_layout()

    # 保存图片
    plt.savefig(
        "glucose_prediction_patient_1.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none",
    )
    plt.show()

    logger.info("\nPrediction plot for Patient 1 saved successfully.")


# Generate the prediction plot
plot_all_predictions(all_results, file_test_data, num_points=600)

In [ ]:
# ============================================================================
# Plot Predictions for Patient 1 (Improved Version)
# ============================================================================


def plot_all_predictions(all_results, file_test_data, num_points=600):
    """
    Plot true glucose and predictions from all models for Patient 1
    with enlarged axis labels and legend positioned outside the plot area
    """

    y_true = file_test_data[1]["y_test"][:num_points]

    # Create figure with extra space for external legend
    fig, ax = plt.subplots(figsize=(24, 12), dpi=300)

    # Plot true glucose values
    ax.plot(
        y_true * NORMALIZE_PARA,
        label="True Glucose",
        color="black",
        linewidth=3.0,
        alpha=0.9,
        zorder=5,
    )

    # Define colors and line styles
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
    linestyles = ["-", "--", "-.", ":", "-", "--"]

    # Plot predictions from each model
    for idx, (model_name, results) in enumerate(all_results.items()):
        y_pred = results["predictions"][0][:num_points]
        ax.plot(
            y_pred * NORMALIZE_PARA,
            label=f"{model_name}",
            color=colors[idx],
            linestyle=linestyles[idx],
            linewidth=2.2,
            alpha=0.75,
            zorder=4 - idx * 0.1,
        )

    # 🔧 设置标题和轴标签 - 大尺寸字体
    ax.set_title(
        f"Glucose Prediction Comparison - Patient 1 Test Set (First {num_points} time points)",
        fontsize=30,
        fontweight="bold",
        pad=30,
    )

    ax.set_xlabel(
        "Time (5-minute intervals)", fontsize=25, fontweight="bold", labelpad=20
    )
    ax.set_ylabel("Glucose (mg/dL)", fontsize=25, fontweight="bold", labelpad=20)

    # 🔧 放大刻度标签 (tick labels)
    ax.tick_params(axis="x", labelsize=40, length=12, width=2.5)
    ax.tick_params(axis="y", labelsize=40, length=12, width=2.5)

    # 🔧 加粗坐标轴线
    for spine in ax.spines.values():
        spine.set_linewidth(3)

    # 🔧 网格设置
    ax.grid(True, alpha=0.3, linestyle="--", linewidth=1.5)

    # 🔧 设置图例到右上外侧 - 避免遮挡
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),  # 图例放在右侧外面
        fontsize=20,
        framealpha=0.95,
        fancybox=True,
        shadow=True,
        frameon=True,
        borderpad=1.5,
    )

    # 设置边距
    plt.margins(x=0.01, y=0.05)

    # 紧凑布局 - 考虑外侧图例
    plt.tight_layout()

    # 保存图片
    plt.savefig(
        "glucose_prediction_patient_1.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none",
    )
    plt.show()

    logger.info("\nPrediction plot for Patient 1 saved successfully.")


# Generate the prediction plot
plot_all_predictions(all_results, file_test_data, num_points=600)

In [ ]:
# ============================================================================
# Plot Predictions for Patient 1 (Improved Version)
# ============================================================================


def plot_all_predictions(all_results, file_test_data, num_points=600):
    """
    Plot true glucose and predictions from all models for Patient 1
    with enlarged axis labels and legend positioned outside the plot area
    """

    y_true = file_test_data[1]["y_test"][:num_points]

    # Create figure with extra space for external legend
    fig, ax = plt.subplots(figsize=(18, 8), dpi=300)

    # Plot true glucose values
    ax.plot(
        y_true * NORMALIZE_PARA,
        label="True Glucose",
        color="black",
        linewidth=3.0,
        alpha=0.9,
        zorder=5,
    )

    # Define colors and line styles
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
    linestyles = ["-", "--", "-.", ":", "-", "--"]

    # Plot predictions from each model
    for idx, (model_name, results) in enumerate(all_results.items()):
        y_pred = results["predictions"][0][:num_points]
        ax.plot(
            y_pred * NORMALIZE_PARA,
            label=f"{model_name}",
            color=colors[idx],
            linestyle=linestyles[idx],
            linewidth=2.0,
            alpha=0.75,
            zorder=4 - idx * 0.1,
        )

    # 🔧 设置标题和轴标签 - 大尺寸字体
    ax.set_title(
        f"Glucose Prediction Comparison - Patient 1 Test Set (First {num_points} time points)",
        fontsize=30,
        fontweight="bold",
        pad=30,
    )

    ax.set_xlabel(
        "Time (5-minute intervals)", fontsize=25, fontweight="bold", labelpad=20
    )
    ax.set_ylabel("Glucose (mg/dL)", fontsize=25, fontweight="bold", labelpad=20)

    # 🔧 放大刻度标签 (tick labels)
    ax.tick_params(axis="x", labelsize=20, length=12, width=2.5)
    ax.tick_params(axis="y", labelsize=20, length=12, width=2.5)

    # 🔧 加粗坐标轴线
    for spine in ax.spines.values():
        spine.set_linewidth(3)

    # 🔧 网格设置
    ax.grid(True, alpha=0.3, linestyle="--", linewidth=1.5)

    # 🔧 设置图例到右上外侧 - 避免遮挡
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),  # 图例放在右侧外面
        fontsize=20,
        framealpha=0.95,
        fancybox=True,
        shadow=True,
        frameon=True,
        borderpad=1.5,
    )

    # 设置边距
    plt.margins(x=0.01, y=0.05)

    # 紧凑布局 - 考虑外侧图例
    plt.tight_layout()

    # 保存图片
    plt.savefig(
        "glucose_prediction_patient_1.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none",
    )
    plt.show()

    logger.info("\nPrediction plot for Patient 1 saved successfully.")


# Generate the prediction plot
plot_all_predictions(all_results, file_test_data, num_points=600)

In [ ]:
# ============================================================================
# Plot Predictions for Patient 1 (Improved Version)
# ============================================================================


def plot_all_predictions(all_results, file_test_data, num_points=600):
    """
    Plot true glucose and predictions from all models for Patient 1
    with enlarged axis labels and legend positioned outside the plot area
    """

    y_true = file_test_data[1]["y_test"][:num_points]

    # Create figure with extra space for external legend
    fig, ax = plt.subplots(figsize=(18, 8), dpi=300)

    # Plot true glucose values
    ax.plot(
        y_true * NORMALIZE_PARA,
        label="True Glucose",
        color="black",
        linewidth=2.0,
        alpha=0.9,
        zorder=5,
    )

    # Define colors and line styles
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
    linestyles = ["-", "--", "-.", ":", "-", "--"]

    # Plot predictions from each model
    for idx, (model_name, results) in enumerate(all_results.items()):
        y_pred = results["predictions"][0][:num_points]
        ax.plot(
            y_pred * NORMALIZE_PARA,
            label=f"{model_name}",
            color=colors[idx],
            linestyle=linestyles[idx],
            linewidth=1.0,
            alpha=0.75,
            zorder=4 - idx * 0.1,
        )

    # 🔧 设置标题和轴标签 - 大尺寸字体
    ax.set_title(
        f"Glucose Prediction Comparison - Patient 1 Test Set (First {num_points} time points)",
        fontsize=25,
        fontweight="bold",
        pad=30,
    )

    ax.set_xlabel(
        "Time (5-minute intervals)", fontsize=20, fontweight="bold", labelpad=20
    )
    ax.set_ylabel("Glucose (mg/dL)", fontsize=20, fontweight="bold", labelpad=20)

    # 🔧 放大刻度标签 (tick labels)
    ax.tick_params(axis="x", labelsize=15, length=12, width=2.5)
    ax.tick_params(axis="y", labelsize=15, length=12, width=2.5)

    # 🔧 加粗坐标轴线
    for spine in ax.spines.values():
        spine.set_linewidth(1)

    # 🔧 网格设置
    ax.grid(True, alpha=0.1, linestyle="--", linewidth=1.5)

    # 🔧 设置图例到右上外侧 - 避免遮挡
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),  # 图例放在右侧外面
        fontsize=15,
        framealpha=0.95,
        fancybox=True,
        shadow=True,
        frameon=True,
        borderpad=1.5,
    )

    # 设置边距
    plt.margins(x=0.01, y=0.05)

    # 紧凑布局 - 考虑外侧图例
    plt.tight_layout()

    # 保存图片
    plt.savefig(
        "glucose_prediction_patient_1.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none",
    )
    plt.show()

    logger.info("\nPrediction plot for Patient 1 saved successfully.")


# Generate the prediction plot
plot_all_predictions(all_results, file_test_data, num_points=600)